# Phase 3: Exploratory Weather & Forecast Error Analysis
### ForecastGuard AI — MoES / NCMRWF (Problem Statement ID: 26079)
**Objective:** Understand NWP forecast distributions, error characteristics, and lead-time dependencies before training bust-detection models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries imported successfully.')

## 1. Load Standardized Forecast Error Dataset
We load `forecast_errors.parquet`, generated in Phase 2 with strict UTC time, SI/meteorological units (°C, mm, m/s, hPa), and Pythagorean wind vector errors.

In [ ]:
df_path = Path('../data/processed/forecast_errors.parquet')
if not df_path.exists():
    df_path = Path('data/processed/forecast_errors.parquet')

df = pd.read_parquet(df_path)
print(f'Total records loaded: {len(df):,}')
print(f'Spatial bounds: Lat [{df.latitude.min()}N, {df.latitude.max()}N], Lon [{df.longitude.min()}E, {df.longitude.max()}E]')
print(f'Lead time span: Day {df.lead_day.min()} to Day {df.lead_day.max()}')
df.head()

## 2. Missing Value & Integrity Audit
Verify complete temporal and spatial coverage without missing values.

In [ ]:
nulls = df.isnull().sum()
print('Missing value count per column:')
print(nulls[nulls > 0] if nulls.sum() > 0 else 'Zero missing values across all records.')

## 3. Forecast vs. Observed Distributions
Comparing predicted vs reference reanalysis/observation distributions across rainfall, temperature, wind, and pressure.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# Rainfall
axes[0, 0].hist(df['forecast_rainfall'], bins=50, alpha=0.6, label='Forecast Rain (mm)', color='blue')
axes[0, 0].hist(df['observed_rainfall'], bins=50, alpha=0.6, label='Observed Rain (mm)', color='green')
axes[0, 0].set_yscale('log')
axes[0, 0].set_title('Rainfall Distribution (Log Scale)')
axes[0, 0].set_xlabel('Rainfall (mm)')
axes[0, 0].legend()

# Temperature
sns.kdeplot(df['forecast_temperature'], ax=axes[0, 1], label='Forecast Temp (°C)', color='red')
sns.kdeplot(df['observed_temperature'], ax=axes[0, 1], label='Observed Temp (°C)', color='orange')
axes[0, 1].set_title('Temperature Kernel Density (°C)')
axes[0, 1].set_xlabel('Temperature (°C)')
axes[0, 1].legend()

# Wind Speed
axes[1, 0].scatter(df['forecast_wind_speed'], df['observed_wind_speed'], alpha=0.1, s=2, color='purple')
axes[1, 0].plot([0, 30], [0, 30], 'r--', label='1:1 Perfect Forecast')
axes[1, 0].set_title('Forecast vs Observed Wind Speed (m/s)')
axes[1, 0].set_xlabel('Forecast Wind Speed (m/s)')
axes[1, 0].set_ylabel('Observed Wind Speed (m/s)')
axes[1, 0].legend()

# Surface Pressure
sns.histplot(df['pressure_error'], bins=40, kde=True, ax=axes[1, 1], color='teal')
axes[1, 1].set_title('Surface Pressure Error Distribution (hPa)')
axes[1, 1].set_xlabel('Error = Forecast - Observed (hPa)')

plt.tight_layout()
plt.show()

## 4. Error Growth by Forecast Lead Day
Crucial meteorological phenomenon: NWP forecast errors grow non-linearly with lead day (Day 1 to Day 10).

In [ ]:
lead_summary = df.groupby('lead_day')[['temperature_absolute_error', 'wind_vector_error', 'combined_error_score']].mean()
print(lead_summary)

plt.figure(figsize=(10, 5))
plt.plot(lead_summary.index, lead_summary['combined_error_score'], marker='o', linewidth=2.5, color='crimson', label='Mean Combined Error Score')
plt.plot(lead_summary.index, lead_summary['wind_vector_error'], marker='s', linewidth=1.5, color='navy', label='Mean Wind Vector Error (m/s)')
plt.plot(lead_summary.index, lead_summary['temperature_absolute_error'], marker='^', linewidth=1.5, color='darkgreen', label='Mean Temp Abs Error (°C)')
plt.title('Forecast Error Progression by Lead Day (Day 1 to Day 10)')
plt.xlabel('Lead Day')
plt.ylabel('Error Metric')
plt.legend()
plt.grid(True)
plt.show()

## 5. Justification of Phase 4 Bust Label Definition
A forecast bust is defined as a forecast whose combined error score exceeds the **90th percentile threshold for its specific lead day**.
This accounts for natural predictability limits while isolating the severe ~10% operational failure cases.